# A first C++ finite element space: P1 triangles

We implement continuous piecewise affine functions with one coefficient per mesh vertex. The local classes `P1TriangleElement` and `P1SegmentElement` are in [myP1TriangleElement.hpp](cpp/myP1TriangleElement.hpp) and [myP1SegmentElement.hpp](cpp/myP1SegmentElement.hpp). Their global space `P1TriangleFESpace` is in [myP1TriangleFESpace.hpp](cpp/myP1TriangleFESpace.hpp). It derives directly from `FESpace` and needs only vertex numbering, explicit shape values and explicit gradients. [mymodule.cpp](cpp/mymodule.cpp) exports the space and the reference-evaluation helpers to Python.

We reproduce the four-triangle computation from Chapter 6, inspect the basis and local-to-global maps, and solve with the existing C++ integrators and assembler. A second problem illustrates nonzero Dirichlet data.

In [ ]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.meshing import Mesh as NetgenMesh, MeshPoint, Pnt, Element2D, Element1D
from teaching_utils import load_module, square_mesh, errors

m = load_module()
np.set_printoptions(precision=6, suppress=True)
affine_subdivisions = 4

## Five vertices and four triangles

Let $\Omega=(0,1)^2$. Join its four corners to the center $z_4=(1/2,1/2)$. We use zero-based numbering.

For

$$
\begin{align*}
-\Delta u=1 \quad\text{in }\Omega,\qquad u=0\quad\text{on }\partial\Omega,
\end{align*}
$$

the space before boundary constraints is

$$
\begin{align*}
S_h=\{v_h\in C(\overline\Omega):v_h|_T\in P_1(T)\},\qquad u_h=\sum_{i=0}^4 U_i\phi_i.
\end{align*}
$$

Only the center coefficient is free.

In [ ]:
coordinates = [(0, 0), (1, 0), (1, 1), (0, 1), (0.5, 0.5)]
triangles = [(0, 1, 4), (1, 2, 4), (2, 3, 4), (3, 0, 4)]

netgen_mesh = NetgenMesh(dim=2)
domain = netgen_mesh.AddRegion("domain", dim=2)
boundary = netgen_mesh.AddRegion("boundary", dim=1)
points = [netgen_mesh.Add(MeshPoint(Pnt(px, py, 0))) for px, py in coordinates]
for vertices in triangles:
    netgen_mesh.Add(Element2D(domain, [points[i] for i in vertices]))
for a, b in [(0, 1), (1, 2), (2, 3), (3, 0)]:
    netgen_mesh.Add(Element1D([points[a], points[b]], index=boundary))

mesh = Mesh(netgen_mesh)
fes = m.P1TriangleFESpace(mesh, dirichlet="boundary")
Draw(mesh, name="Four P1 triangles");

## The local element: values and derivatives

The reference vertices are $(1,0),(0,1),(0,0)$. The class `P1TriangleElement` evaluates the basis

$$
\begin{align*}
(\widehat\phi_0,\widehat\phi_1,\widehat\phi_2)=(\xi,\eta,1-\xi-\eta).
\end{align*}
$$

Its `CalcShape` method evaluates these three functions. Its `CalcDShape` method returns their reference gradients as rows

$$
\begin{align*}
D_{\mathrm{ref}}=\begin{pmatrix}1&0\\0&1\\-1&-1\end{pmatrix}.
\end{align*}
$$


In [ ]:
first_cell = next(iter(mesh.Elements()))
element = fes.GetFE(ElementId(VOL, first_cell.nr))
reference_nodes = [(1, 0), (0, 1), (0, 0)]
reference_values = np.array([element.CalcShape(px, py) for px, py in reference_nodes])
reference_gradients = np.array(element.CalcDShape(0.2, 0.3))

print("Basis values at the reference vertices:")
print(reference_values)
print(f"Basis values at (0.2, 0.3): {np.array(element.CalcShape(0.2, 0.3))}")
print("Reference gradients, one row per basis function:")
print(reference_gradients)

## The global space: connect coefficients across elements

The three central methods in `P1TriangleFESpace` have different roles:

| Method | Task in this P1 space |
|---|---|
| `Update` | Set the total number of coefficients to the number of mesh vertices. |
| `GetFE` | Supply a `P1TriangleElement` for a volume cell or a `P1SegmentElement` for a boundary edge. |
| `GetDofNrs` | Return the global vertex numbers in the local element's vertex order. |

Thus

$$
\begin{align*}
u_h|_T=\sum_{\alpha=0}^2 U_{g_T(\alpha)}\phi_\alpha^T.
\end{align*}
$$

Neighboring triangles use the same coefficient at a shared vertex. Their linear edge traces then agree because their endpoint values agree.

The boundary segment has endpoint functions $(\xi,1-\xi)$ in the local order $1,0$. Its numbering also identifies mesh vertices. This supplies the boundary trace used when imposing Dirichlet data.

In [ ]:
print(f"Global coefficients: {fes.ndof}")
print(f"Free coefficient numbers: {np.flatnonzero(np.array(list(fes.FreeDofs())))}")
for cell in mesh.Elements():
    vertices = [vertex.nr for vertex in cell.vertices]
    print(f"T{cell.nr}: vertices {vertices}, dofs {list(fes.GetDofNrs(cell))}")
for edge in mesh.Elements(BND):
    print(f"Boundary edge {edge.nr}: dofs {list(fes.GetDofNrs(edge))}")

## From the reference triangle to the bottom triangle $T_0$

The local vertices of $T_0$ are $(z_0,z_1,z_4)$. The affine map $F_{T_0}:\widehat T\to T_0$ sends reference coordinates to physical coordinates

$$
\begin{align*}
F_{T_0}(\xi,\eta)=z_4+J_{T_0}\begin{pmatrix}\xi\\\eta\end{pmatrix},
\qquad J_{T_0}=\begin{pmatrix}-1/2&1/2\\-1/2&-1/2\end{pmatrix}.
\end{align*}
$$

Thus $|\det J_{T_0}|=1/2$ and $|T_0|=|\det J_{T_0}|\,|\widehat T|=1/4$. The physical basis functions are defined by $\phi_\alpha^{T_0}=\widehat\phi_\alpha\circ F_{T_0}^{-1}$. At corresponding points, the chain rule gives

$$
\begin{align*}
\nabla\phi_\alpha^{T_0}(F_{T_0}(\widehat x))=J_{T_0}^{-\top}\widehat\nabla\widehat\phi_\alpha(\widehat x).
\end{align*}
$$

The manuscript's first example collects gradients as columns in $\widehat G$ and $G$. Here and in C++, they are rows: $D_{\mathrm{ref}}=\widehat G^\top$ and $D_x=G^\top$. Transposing the chain rule therefore gives

$$
\begin{align*}
D_x=D_{\mathrm{ref}}J_{T_0}^{-1}.
\end{align*}
$$

We evaluate the reference basis at reference quadrature points. The map supplies the geometry and transforms gradients and integration weights.

In [ ]:
points_T0 = np.array([mesh.vertices[vertex.nr].point[:2] for vertex in first_cell.vertices])
J_T0 = np.column_stack((points_T0[0]-points_T0[2], points_T0[1]-points_T0[2]))
det_T0 = abs(np.linalg.det(J_T0))
area_T0 = det_T0/2

print(f"J_T0:\n{J_T0}")
print(f"|det J_T0| = {det_T0}, area |T0| = {area_T0}")

## Element matrices and load vectors

Changing variables from $T_0$ to $\widehat T$ gives

$$
\begin{align*}
A_{T_0}&=|\det J_{T_0}|\int_{\widehat T}D_xD_x^\top\,d\widehat x =\frac12\begin{pmatrix}1&0&-1\\0&1&-1\\-1&-1&2\end{pmatrix},\\
b_{T_0}&=|\det J_{T_0}|\int_{\widehat T}\begin{pmatrix}\xi\\\eta\\1-\xi-\eta\end{pmatrix}\,d\widehat x=\frac1{12}\begin{pmatrix}1\\1\\1\end{pmatrix}.
\end{align*}
$$

For the constant stiffness integrand, $\int_{\widehat T}1\,d\widehat x=1/2$. The load integrals are

$$
\begin{align*}
\int_0^1\int_0^{1-\xi}\xi\,d\eta\,d\xi&=\int_0^1\xi(1-\xi)\,d\xi=\frac16,\\
\int_0^1\int_0^{1-\xi}\eta\,d\eta\,d\xi&=\frac12\int_0^1(1-\xi)^2\,d\xi=\frac16,\\
\int_0^1\int_0^{1-\xi}(1-\xi-\eta)\,d\eta\,d\xi&=\frac12\int_0^1(1-\xi)^2\,d\xi=\frac16.
\end{align*}
$$

A quadrature rule exact for linear polynomials reproduces these integrals. For this affine $P_1$ example with diffusion coefficient and source equal to one, a single reference point $\widehat x_q=(1/3,1/3)$ with reference weight $w_q=1/2$ suffices:

$$
\begin{align*}
\omega_q&=w_q|\det J_{T_0}|=\frac14,\\
A_{T_0}&=\omega_q D_xD_x^\top,\qquad b_{T_0}=\omega_q\widehat\Phi(\widehat x_q), \quad \widehat\Phi=(\widehat\phi_0,\widehat\phi_1,\widehat\phi_2)^\top.
\end{align*}
$$

The next cell follows the C++ integrators: `CalcShape` and `CalcDShape` evaluate at the reference point, multiplication by $J_{T_0}^{-1}$ transforms the gradient rows, and $w_q|\det J_{T_0}|$ is the physical integration weight. In C++, these geometry factors come from `mip.GetJacobianInverse()` and `mip.GetMeasure()`. The default C++ quadrature has exactness order $2p+2=4$.

The other triangles are rotations of $T_0$. With the local lists

$$
\begin{align*}
g_{T_0}=(0,1,4),\quad g_{T_1}=(1,2,4),\quad g_{T_2}=(2,3,4),\quad g_{T_3}=(3,0,4),
\end{align*}
$$

all four local matrices and load vectors have the same entries. We first evaluate the one-point rule for $T_0$, then call the actual C++ integrators on every element. Displaying $2A_T$ and $12b_T$ makes the integer entries from the manuscript visible.

In [ ]:
reference_point_T0 = (1/3, 1/3)
reference_weight_T0 = 0.5
shape_T0 = np.array(element.CalcShape(*reference_point_T0))
reference_gradients_T0 = np.array(element.CalcDShape(*reference_point_T0))
physical_gradients_T0 = reference_gradients_T0 @ np.linalg.inv(J_T0)
quadrature_weight_T0 = reference_weight_T0 * det_T0

A_T0_hand = quadrature_weight_T0 * (physical_gradients_T0 @ physical_gradients_T0.T)
b_T0_hand = quadrature_weight_T0 * shape_T0
print(f"Reference point: {reference_point_T0}, reference weight: {reference_weight_T0}")
print(f"Reference basis values: {shape_T0}")
print(f"Physical gradients (rows):\n{physical_gradients_T0}")
print(f"Physical integration weight: {quadrature_weight_T0}")
print(f"One-point quadrature, 2 A_T0:\n{2*A_T0_hand}")
print(f"One-point quadrature, 12 b_T0: {12*b_T0_hand}")

laplace = m.Laplace(CF(1))
source = m.Source(CF(1))
local_dofs, local_matrices, local_loads = [], [], []
for cell in mesh.Elements():
    ei = ElementId(VOL, cell.nr)
    fe = fes.GetFE(ei)
    trafo = mesh.GetTrafo(ei)
    dofs = list(fes.GetDofNrs(cell))
    A_T = np.array(laplace.CalcElementMatrix(fe, trafo))
    b_T = np.array(source.CalcElementVector(fe, trafo))
    local_dofs.append(dofs)
    local_matrices.append(A_T)
    local_loads.append(b_T)
    print(f"\nT{cell.nr}, g_T = {dofs}")
    print(f"2 A_T:\n{2*A_T}")
    print(f"12 b_T: {12*b_T}")

## Physical basis functions as a check

The transformed gradients also let us reconstruct the physical affine functions

$$
\begin{align*}
\phi_0^{T_0}=1-x-y,\qquad \phi_1^{T_0}=x-y,\qquad \phi_2^{T_0}=2y.
\end{align*}
$$

This provides an additional check of the mapping and local ordering. The integration above uses reference evaluations and the geometry factors directly.

For $\phi_\alpha^{T_0}(x,y)=c_\alpha+\partial_x\phi_\alpha^{T_0}\,x+\partial_y\phi_\alpha^{T_0}\,y$, the following table has rows $(c_\alpha,\partial_x\phi_\alpha^{T_0},\partial_y\phi_\alpha^{T_0})$. Since $F_{T_0}(0,0)=z_4$, the constant terms follow from the reference values at $(0,0)$.


In [ ]:
constants_T0 = np.array(element.CalcShape(0, 0)) - physical_gradients_T0 @ points_T0[2]
physical_coefficients_T0 = np.column_stack((constants_T0, physical_gradients_T0))
print("Physical basis coefficients [constant, d/dx, d/dy]:")
print(physical_coefficients_T0)

## Assemble on the full space

For $\mathcal A(u,v)=\int_\Omega\nabla u\cdot\nabla v\,dx$ and $\ell(v)=\int_\Omega v\,dx$, each local contribution is added at its global indices

$$
\begin{align*}
A_{g_T(\alpha),g_T(\beta)}\mathrel{+}=(A_T)_{\alpha\beta}, \qquad b_{g_T(\alpha)}\mathrel{+}=(b_T)_\alpha.
\end{align*}
$$

Local index $2$ is the center on every triangle. Its four diagonal contributions add to $A_{44}=1+1+1+1=4$. Each corner receives two load contributions $1/12$, while the center receives four. The manuscript therefore obtains

$$
\begin{align*}
A=\begin{pmatrix}
1&0&0&0&-1\\
0&1&0&0&-1\\
0&0&1&0&-1\\
0&0&0&1&-1\\
-1&-1&-1&-1&4
\end{pmatrix},
\qquad
b=\frac16\begin{pmatrix}1\\1\\1\\1\\2\end{pmatrix}.
\end{align*}
$$

The explicit loops below assemble the four computed local contributions. We then compare their result with the existing global C++ assembler.

In [ ]:
A_scatter = np.zeros((fes.ndof, fes.ndof))
b_scatter = np.zeros(fes.ndof)
for k, (dofs, A_T, b_T) in enumerate(zip(local_dofs, local_matrices, local_loads)):
    for alpha, i in enumerate(dofs):
        b_scatter[i] += b_T[alpha]
        for beta, j in enumerate(dofs):
            A_scatter[i, j] += A_T[alpha, beta]
    print(f"After T{k}: A[4,4] = {A_scatter[4,4]:g}, 12*b[4] = {12*b_scatter[4]:g}")

A = m.AssembleMatrix(fes, [laplace])
b = m.AssembleVector(fes, [source])
A_dense = np.array(A.ToDense())
b_values = b.FV().NumPy().copy()

print(f"\nFull matrix from the explicit scatter loop:\n{A_scatter}")
print(f"Full matrix from the C++ assembler:\n{A_dense}")
print(f"6 b from the scatter loop: {6*b_scatter}")
print(f"6 b from the C++ assembler: {6*b_values}")
print(f"Largest matrix difference: {np.max(np.abs(A_scatter-A_dense))}")
print(f"Largest load difference: {np.max(np.abs(b_scatter-b_values))}")

## Impose the boundary values and solve

The four corner coefficients are constrained to zero. The remaining equation is

$$
\begin{align*}
A_{44}U_4=b_4,\qquad 4U_4=\frac13.
\end{align*}
$$

Hence $U_4=1/12$ and $u_h=\phi_4/12$.

`FreeDofs()` selects the free block for the sparse solve. The full matrix has the constant vector in its kernel, but the restricted matrix is positive definite.

In [ ]:
free = np.flatnonzero(np.array(list(fes.FreeDofs())))
A_FF = A_dense[np.ix_(free, free)]
b_F = b_values[free]
U_F = np.linalg.solve(A_FF, b_F)

print(f"Free indices: {free}")
print(f"A_FF: {A_FF}")
print(f"3 b_F: {3*b_F} (so b_F = 1/3)")
print(f"12 U_F: {12*U_F} (so U_4 = 1/12)")

uh = GridFunction(fes)
uh.vec.data = A.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * b
patch_values = uh.vec.FV().NumPy().copy()
print(f"Full coefficient vector U: {patch_values}")
print(f"12 U: {12*patch_values}")
Draw(uh, mesh, name="P1 Galerkin solution", deformation=True);

## Nonzero Dirichlet data

On a finer triangular mesh, choose the affine function

$$
\begin{align*}
g(x,y)=1+x+2y,\qquad -\Delta g=0.
\end{align*}
$$

Its boundary trace and the function itself are exactly representable in the P1 space.

Set the prescribed boundary coefficients in an initial vector $G$, then solve for a correction that vanishes on the boundary

$$
\begin{align*}
A_{FF}\,\delta U_F=(b-AG)_F,\qquad \delta U_D=0.
\end{align*}
$$

The final coefficient vector is $U=G+\delta U$. This is the discrete lifting procedure from the manuscript.

In [ ]:
affine_mesh = square_mesh(affine_subdivisions)
affine_fes = m.P1TriangleFESpace(affine_mesh, dirichlet=".*")
g = 1 + x + 2*y
affine_A = m.AssembleMatrix(affine_fes, [m.Laplace(CF(1))])
affine_b = m.AssembleVector(affine_fes, [m.Source(CF(0))])

affine_uh = GridFunction(affine_fes)
affine_uh.Set(g, definedon=affine_mesh.Boundaries(".*"))
rhs = affine_b.CreateVector()
rhs.data = affine_b - affine_A*affine_uh.vec
affine_uh.vec.data += affine_A.Inverse(affine_fes.FreeDofs(), inverse="sparsecholesky") * rhs

affine_errors = errors(affine_mesh, affine_uh, g)
print(f"L2 error: {affine_errors['L2']}")
print(f"H1 error: {affine_errors['H1']}")

Continue with [nodal and hierarchical higher-order bases](01_nodal_elements.ipynb) and [local integration and global assembly](02_assembly.ipynb).

[Chapter 6 notebooks](README.md)